# RQ1 — Effectiveness: Does LLM Guidance Improve Learning?

**Research question:** Does LLM-guided policy distillation (LLM4Teach) yield better penetration-testing agents than a standard PPO baseline trained without any teacher signal?

**Conditions compared**
| Label | Folder | Description |
|---|---|---|
| `llm_full` (local) | `rq1/tiny/llm_full` | Full system — local Qwen-4B teacher |
| `llm_full` (HPC) | _configurable_ | Full system — HPC teacher (add path when available) |
| `ppo_options` | `rq1/tiny/ppo_options` | PPO with option hierarchy, **no LLM teacher** |
| `random` | _synthetic_ | Random action selection (lower bound) |
| `bruteforce` | _synthetic_ | Exhaustive search (upper-performance reference) |

**Data:** `train.csv` per run — one row per training episode.  
**Step logs:** `step_logs/episode_N.txt` — per-step action/outcome text for qualitative analysis.

In [ ]:
"""Imports and global configuration — RQ1."""
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from scipy import stats

# ── Resolve project root ────────────────────────────────────────────────────
import os as _os

def _find_project_root():
    # Primary: use the installed nasim package to locate the project directory
    try:
        import nasim as _n
        return Path(_n.__file__).resolve().parent.parent
    except ImportError:
        pass
    # Fallback: walk up from CWD searching for the nasim sub-directory
    _cwd = Path(_os.getcwd()).resolve()
    for _p in [_cwd] + list(_cwd.parents):
        if (_p / "nasim").is_dir():
            return _p
    return _cwd

_project_root = _find_project_root()
del _find_project_root

# ── Output directory ──────────────────────────────────────────────────────────
FIGURES_DIR = _project_root / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RUNS_ROOT = _project_root / "runs"

# ── Scenario under evaluation ─────────────────────────────────────────────────
# Path structure expected:  runs/{run tag}/rq1/{SCENARIO}/{condition}/seed{N}/train.csv
SCENARIO = "tiny"   # change to e.g. "small" or "medium" for a different scenario

# ── Runs to compare (llm_full experiments) ────────────────────────────────────
# Each entry is one experiment run you want to appear as an llm_full line.
# Add or remove entries freely — the plots loop over this list automatically.
#
# Required keys:
#   tag   : subfolder name under runs/
#   label : legend display name (you decide)
#   color : matplotlib color string
#   ls    : linestyle ("-", "--", "-.", ":")
RUNS = [
    {
        "tag":   "qwen-4B_test",
        "label": "Qwen-4B",
        "color": "#0072B2",   # Okabe-Ito blue
        "ls":    "-",
    },
    # Add further runs here, for example:
    # {
    #     "tag":   "llama-3B_hpc",
    #     "label": "LLaMA-3B",
    #     "color": "#56B4E9",   # Okabe-Ito sky-blue
    #     "ls":    "--",
    # },
]

# ── No-LLM baseline style (ppo_options) ───────────────────────────────────────
# Looked up from whichever run in RUNS has ppo_options data.
PPO_STYLE = {
    "color": "#E69F00",            # Okabe-Ito orange
    "ls":    "-",
    "label": "PPO + Options (no LLM)",
}

# ── Reference baselines (random / bruteforce) ─────────────────────────────────
# Populate when evaluation data for those agents is available.
# Option A — scalar: set success_rate (0–1) and/or reward (float) directly.
# Option B — CSV:    set csv_path; the notebook reads the last 10 episodes.
# Leave a field None to skip it.
BASELINES = {
    "random": {
        "label":        "Random baseline",
        "color":        "#BBBBBB",
        "ls":           ":",
        "success_rate": None,   # e.g. 0.0
        "reward":       None,   # e.g. -80.0
        "csv_path":     None,   # e.g. Path("../runs/qwen-4B_test/tiny/rq1/random/seed0/train.csv")
    },
    "bruteforce": {
        "label":        "Bruteforce reference",
        "color":        "#555555",
        "ls":           ":",
        "success_rate": None,   # e.g. 1.0
        "reward":       None,   # e.g. 220.0
        "csv_path":     None,
    },
}

SEEDS = list(range(10))   # probed 0..9; missing seeds are silently skipped

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi":      150,
    "figure.figsize":  (10, 5),
    "font.size":       12,
    "axes.titlesize":  14,
    "axes.labelsize":  13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "axes.grid":       True,
    "grid.alpha":      0.3,
    "lines.linewidth": 2.0,
})

# ── Startup report ────────────────────────────────────────────────────────────
print("Configuration loaded.")
print(f"  Scenario  : {SCENARIO}")
print(f"  Runs      : {[r['tag'] for r in RUNS]}")
print(f"  Figures   : {FIGURES_DIR.resolve()}")
for bkey, bval in BASELINES.items():
    if bval["success_rate"] is None and bval["csv_path"] is None:
        print(f"  [WARN] Baseline '{bkey}' not yet populated — fill BASELINES above.")

## 1. Data Discovery & Schema Inspection

Load every available `train.csv`, print column names, dtypes, a sample, and unique values of
key categorical columns. This cell must run cleanly before any plotting code.

In [ ]:
def _load_run_conditions(run_tag: str, scenario: str, conditions: list, seeds: list) -> dict:
    """Load train.csv files for one run tag.

    Returns {condition: {seed: DataFrame}} for that run.
    """
    base   = RUNS_ROOT / run_tag / "rq1" / scenario
    result = {}
    for cond in conditions:
        result[cond] = {}
        for seed in seeds:
            csv = base / cond / f"seed{seed}" / "train.csv"
            if csv.exists():
                df = pd.read_csv(csv)
                if len(df) > 0:
                    df["seed"]      = seed
                    df["condition"] = cond
                    df["run"]       = run_tag
                    result[cond][seed] = df
                else:
                    print(f"  [WARN] {csv.relative_to(RUNS_ROOT)} — 0 rows, skipping")
    return result


CONDITIONS = ["llm_full", "ppo_options"]

# all_data[run_tag][condition] = {seed: DataFrame}
all_data = {}
for run in RUNS:
    all_data[run["tag"]] = _load_run_conditions(run["tag"], SCENARIO, CONDITIONS, SEEDS)

# ── Schema inspection ──────────────────────────────────────────────────────────
print("=" * 70)
print("SCHEMA INSPECTION — train.csv")
print("=" * 70)
sample_df = None
for run in RUNS:
    for cond in CONDITIONS:
        sd = all_data[run["tag"]].get(cond, {})
        if sd:
            sample_df = next(iter(sd.values()))
            print(f"\nSample: run={run['tag']}  condition={cond}  seed={next(iter(sd))}")
            break
    if sample_df is not None:
        break

if sample_df is not None:
    print(f"\nShape  : {sample_df.shape}")
    print(f"\nColumn dtypes:")
    print(sample_df.dtypes.to_string())
    print(f"\nFirst 3 rows:")
    drop_cols = ["seed", "condition", "run"]
    print(sample_df.drop(columns=drop_cols, errors="ignore").head(3).to_string())
    print(f"\nUnique 'success' values: {sorted(sample_df['success'].unique())}")

# ── Availability summary ───────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("AVAILABILITY SUMMARY")
print("=" * 70)
for run in RUNS:
    for cond in CONDITIONS:
        sd          = all_data[run["tag"]].get(cond, {})
        seeds_found = sorted(sd.keys())
        ep_counts   = [len(sd[s]) for s in seeds_found]
        print(f"  {run['tag']:20s}  {cond:15s}  seeds={seeds_found}  ep={ep_counts}")
print("=" * 70)

### Column Mapping Notes

The table below maps the names used in this notebook to the actual column names found in `train.csv`.

| Notebook concept | Actual column | Notes |
|---|---|---|
| Episode index | `episode` | 1-based integer |
| **Shaped** episode reward | `reward` | Includes step penalties and KL-divergence penalty terms used to steer training. **Not the same as native task reward.** Used only as a secondary diagnostic. |
| Smoothed shaped reward | `avg_reward_10` | Pre-computed 10-episode rolling mean of `reward`, logged by the training script |
| **Native** episode success | `success` | Binary 0/1 — did the agent fully compromise all sensitive targets? No penalty terms. **This is the primary evaluation metric throughout this notebook.** |
| Compromise rate | `compromise_rate` | Fraction of all hosts compromised; secondary progress signal |
| Episode length | `length` | Number of environment steps taken |
| Teacher agreement | `teacher_agree` | % of steps where the agent matched the LLM recommendation (0 for no-LLM runs) |

> **Why success rate, not reward?**  
> The `reward` column is the *shaped* training signal. It folds in step-count penalties and,
> for LLM4Teach runs, KL-divergence terms that regularise the policy toward the teacher
> distribution. These extra terms have different magnitudes across conditions and scenarios,
> making cross-condition reward comparisons misleading. The `success` column (did the agent
> reach every sensitive target?) is free of those artefacts. All headline figures use `success`;
> `reward` appears only as a secondary diagnostic to show the training signal the agent actually
> optimised.

In [ ]:
"""Step-log discovery and structure inspection (uses first run in RUNS)."""

def load_step_log(run_tag: str, condition: str, seed: int, episode: int) -> list[str]:
    path = (RUNS_ROOT / run_tag / "rq1" / SCENARIO
            / condition / f"seed{seed}" / "step_logs" / f"episode_{episode}.txt")
    if path.exists():
        return path.read_text(encoding="utf-8", errors="replace").splitlines()
    return []

print("=" * 70)
print("STEP-LOG STRUCTURE — first available episode")
print("=" * 70)
printed = False
for run in RUNS:
    if printed:
        break
    for cond in CONDITIONS:
        if printed:
            break
        for seed in SEEDS:
            lines = load_step_log(run["tag"], cond, seed, episode=1)
            if lines:
                print(f"Source: runs/{run['tag']}/rq1/{SCENARIO}/{cond}/seed{seed}/step_logs/episode_1.txt")
                print(f"Total lines: {len(lines)}\n")
                print("First 25 lines:")
                print("\n".join(lines[:25]))
                printed = True
                break

if not printed:
    print("No step-log files found across any configured run.")

## 2. Helper Functions

In [ ]:
def mean_and_band(seed_dict: dict, metric: str, window: int = 10):
    """Return (episodes, mean, lower, upper) arrays aligned across seeds.

    If only one seed is available, returns a rolling-mean smoothed line
    and None for lower/upper (no band can be drawn).

    Parameters
    ----------
    seed_dict : {seed_int: DataFrame}
    metric    : column name in the DataFrames
    window    : rolling-mean window when a single seed is present
    """
    if not seed_dict:
        return None, None, None, None

    # Align on the minimum episode count across seeds
    all_dfs = list(seed_dict.values())
    min_ep  = min(len(df) for df in all_dfs)
    # Stack into (n_seeds, n_episodes) matrix
    matrix = np.stack([df[metric].values[:min_ep] for df in all_dfs], axis=0)
    episodes = np.arange(1, min_ep + 1)

    if len(all_dfs) == 1:
        # Single seed: smooth with rolling mean, note no CI available
        raw = matrix[0]
        s   = pd.Series(raw).rolling(window, min_periods=1, center=True).mean().values
        return episodes, s, None, None
    else:
        # Multiple seeds: mean ± 95% CI
        mu  = matrix.mean(axis=0)
        se  = stats.sem(matrix, axis=0)
        ci  = se * stats.t.ppf(0.975, df=len(all_dfs) - 1)
        return episodes, mu, mu - ci, mu + ci


def save_figure(fig, name: str):
    """Save figure as PNG (300 dpi) and PDF to FIGURES_DIR."""
    png = FIGURES_DIR / f"{name}.png"
    pdf = FIGURES_DIR / f"{name}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"  Saved: {png}")
    print(f"  Saved: {pdf}")


print("Helper functions defined.")

## 3. Figure 1 — Success Rate Over Training  *(primary evaluation figure)*

**What this shows:** Task success rate (fraction of episodes in which the agent fully compromised
all sensitive targets) over training, one line per method. This is the native task signal — no
step penalties or KL terms — and is the primary metric for the thesis claim.  
The LLM4Teach line should climb faster and reach a higher plateau than PPO+Options, demonstrating
that LLM guidance translates into better task completion, not just a higher shaped training reward.

Shaded bands show ±95% confidence intervals across seeds; when only one seed is available the
line is smoothed with a 10-episode rolling mean (noted in the legend).  
`random` (lower bound) and `bruteforce` (upper bound) appear as horizontal reference lines once
populated in `BASELINES` above.

In [ ]:
"""Figure 1 — Success-rate learning curve (PRIMARY evaluation figure).

Metric   : `success` (0/1) — native task signal, no shaped penalties.
One line per run in RUNS (llm_full condition) + one line for ppo_options.
Baselines: horizontal lines from BASELINES if populated.
"""

def _resolve_baseline_success(bval: dict, window: int = 10):
    if bval.get("success_rate") is not None:
        return float(bval["success_rate"])
    if bval.get("csv_path") is not None:
        p = Path(bval["csv_path"])
        if p.exists():
            df = pd.read_csv(p)
            if len(df) > 0 and "success" in df.columns:
                return df["success"].tail(window).mean()
            print(f"  [WARN] Baseline CSV unusable: {p}")
        else:
            print(f"  [WARN] Baseline CSV not found: {p}")
    return None


fig, ax = plt.subplots(figsize=(11, 5.5))
any_plotted        = False
single_seed_warning = False

# ── One line per run (llm_full) ────────────────────────────────────────────────
for run in RUNS:
    sd = all_data[run["tag"]].get("llm_full", {})
    if not sd:
        print(f"  [SKIP] llm_full: no data for run '{run['tag']}'")
        continue
    eps, mu, lo, hi = mean_and_band(sd, "success")
    lbl = run["label"]
    if len(sd) == 1:
        lbl += " (1 seed, rolling mean)"
        single_seed_warning = True
    ax.plot(eps, mu * 100, color=run["color"], linestyle=run["ls"],
            linewidth=2.2, label=lbl, zorder=3)
    if lo is not None:
        ax.fill_between(eps, lo * 100, hi * 100, alpha=0.18, color=run["color"], zorder=2)
    any_plotted = True

# ── PPO baseline (first run that has it) ──────────────────────────────────────
for run in RUNS:
    sd = all_data[run["tag"]].get("ppo_options", {})
    if not sd:
        continue
    eps, mu, lo, hi = mean_and_band(sd, "success")
    lbl = PPO_STYLE["label"]
    if len(sd) == 1:
        lbl += " (1 seed, rolling mean)"
        single_seed_warning = True
    ax.plot(eps, mu * 100, color=PPO_STYLE["color"], linestyle=PPO_STYLE["ls"],
            linewidth=2.2, label=lbl, zorder=3)
    if lo is not None:
        ax.fill_between(eps, lo * 100, hi * 100, alpha=0.18,
                        color=PPO_STYLE["color"], zorder=2)
    any_plotted = True
    break   # only draw once

# ── Reference baselines ────────────────────────────────────────────────────────
for bval in BASELINES.values():
    val = _resolve_baseline_success(bval)
    if val is not None:
        ax.axhline(val * 100, color=bval["color"], linestyle=bval["ls"],
                   linewidth=1.8, label=bval["label"], zorder=1)

ax.set_xlabel("Training Episode", fontsize=13, fontweight="bold")
ax.set_ylabel("Success Rate (%)", fontsize=13, fontweight="bold")
ax.set_title(f"RQ1 — Task Success Rate Over Training  [{SCENARIO} scenario]",
             fontsize=15, fontweight="bold", pad=12)
ax.set_ylim([-5, 105])
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=0))
ax.legend(loc="lower right", framealpha=0.9)

if single_seed_warning:
    ax.text(0.01, 0.01,
            "Note: only 1 seed — line is a 10-ep rolling mean, no CI.",
            transform=ax.transAxes, fontsize=9, color="#555555",
            va="bottom", ha="left")
if not any_plotted:
    ax.text(0.5, 0.5, "No data found — check RUNS_ROOT and SCENARIO.",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=13, color="red")

plt.tight_layout()
save_figure(fig, "rq1_success_rate")
plt.show()

## 4. Figure 2 — Sample Efficiency (Horizontal Bar Chart)

**What this shows:** For each method, the number of training episodes required to first sustain a
fixed **success-rate** threshold (10-episode rolling mean ≥ threshold). Thresholding on success
rate rather than shaped reward avoids the bias from KL and step penalties that differ across
conditions.

The threshold defaults to 50% success (i.e. the agent succeeds in at least half of recent
episodes), which is a meaningful first milestone in penetration testing. Set
`SUCCESS_THRESHOLD_OVERRIDE` to a different value (0–1) if a scenario-specific milestone is
more appropriate. A shorter bar means faster learning — the core sample-efficiency claim.

In [ ]:
"""Figure 2 — Sample-efficiency horizontal bar chart.

Metric   : `success` — native signal, no shaped penalties.
Threshold: first episode where 10-ep rolling success rate ≥ SUCCESS_THRESHOLD.
"""

SUCCESS_THRESHOLD_OVERRIDE = None   # override with e.g. 0.80

success_threshold = float(SUCCESS_THRESHOLD_OVERRIDE) if SUCCESS_THRESHOLD_OVERRIDE else 0.50
print(f"Success-rate threshold: {success_threshold * 100:.0f}%")

ROLLING_W  = 10
efficiency = {}   # label -> (mean_first_ep, ci_half)

def _first_crossing(sd: dict, threshold: float, rolling_w: int):
    """Return (mean_episode, ci) across seeds for the first crossing, or (None, 0)."""
    crossings = []
    for df in sd.values():
        rolled = df["success"].rolling(rolling_w, min_periods=1).mean()
        above  = rolled >= threshold
        if above.any():
            crossings.append(int(df.loc[above.idxmax(), "episode"]))
        else:
            crossings.append(None)
    valid   = [c for c in crossings if c is not None]
    n_never = len(crossings) - len(valid)
    return valid, n_never


# llm_full lines (one per run)
for run in RUNS:
    sd = all_data[run["tag"]].get("llm_full", {})
    if not sd:
        continue
    valid, n_never = _first_crossing(sd, success_threshold, ROLLING_W)
    if n_never:
        print(f"  [{run['label']}] {n_never}/{len(sd)} seeds never reached threshold")
    if not valid:
        efficiency[run["label"]] = (None, 0.0, run["color"])
    elif len(valid) == 1:
        efficiency[run["label"]] = (valid[0], 0.0, run["color"])
    else:
        mu = np.mean(valid)
        ci = stats.sem(valid) * stats.t.ppf(0.975, len(valid) - 1)
        efficiency[run["label"]] = (mu, ci, run["color"])

# ppo_options (first run that has it)
for run in RUNS:
    sd = all_data[run["tag"]].get("ppo_options", {})
    if not sd:
        continue
    valid, n_never = _first_crossing(sd, success_threshold, ROLLING_W)
    lbl = PPO_STYLE["label"]
    if n_never:
        print(f"  [{lbl}] {n_never}/{len(sd)} seeds never reached threshold")
    if not valid:
        efficiency[lbl] = (None, 0.0, PPO_STYLE["color"])
    elif len(valid) == 1:
        efficiency[lbl] = (valid[0], 0.0, PPO_STYLE["color"])
    else:
        mu = np.mean(valid)
        ci = stats.sem(valid) * stats.t.ppf(0.975, len(valid) - 1)
        efficiency[lbl] = (mu, ci, PPO_STYLE["color"])
    break

# ── Draw chart ────────────────────────────────────────────────────────────────
eff_items = [(lbl, v) for lbl, v in efficiency.items() if v[0] is not None]
eff_items.sort(key=lambda x: x[1][0])

if not eff_items:
    print(f"No method reached {success_threshold*100:.0f}% success. "
          "Lower SUCCESS_THRESHOLD_OVERRIDE.")
else:
    labels_bar = [lbl for lbl, _ in eff_items]
    means_bar  = [v[0] for _, v in eff_items]
    cis_bar    = [v[1] for _, v in eff_items]
    colors_bar = [v[2] for _, v in eff_items]

    fig, ax = plt.subplots(figsize=(9, max(2.5, 0.8 * len(eff_items) + 1.5)))
    y    = np.arange(len(eff_items))
    bars = ax.barh(y, means_bar, xerr=cis_bar, color=colors_bar,
                   edgecolor="black", linewidth=0.8, capsize=4, height=0.5, zorder=3)

    max_ci = max(cis_bar) if cis_bar else 0
    for bar, val, ci in zip(bars, means_bar, cis_bar):
        lstr = f"{val:.0f} ep" + (f" ±{ci:.0f}" if ci > 0 else "")
        ax.text(bar.get_width() + max_ci * 0.05 + 0.5,
                bar.get_y() + bar.get_height() / 2,
                lstr, va="center", fontsize=11, fontweight="bold")

    ax.set_yticks(y)
    ax.set_yticklabels(labels_bar, fontsize=12)
    ax.set_xlabel("Episodes to First Reach Threshold", fontsize=13, fontweight="bold")
    ax.set_title(
        f"RQ1 — Sample Efficiency\n"
        f"(threshold = {success_threshold*100:.0f}% success rate, 10-ep rolling mean)",
        fontsize=14, fontweight="bold", pad=10)
    ax.invert_yaxis()
    ax.grid(True, axis="x", alpha=0.4)
    ax.set_axisbelow(True)

    plt.tight_layout()
    save_figure(fig, "rq1_sample_efficiency")
    plt.show()

## 5. Figure 3 — Shaped Reward Over Training  *(secondary diagnostic)*

**What this shows:** The `reward` column — the shaped training signal the policy actually
optimised. This is a *diagnostic*, not an evaluation. Because `reward` includes KL-divergence
penalties (which are larger for the LLM-guided runs) and step-count penalties, it is **not
comparable across conditions at face value**. It is plotted here to confirm the optimiser was
making progress and to expose training instabilities, not to rank methods.

The primary evaluation figure is Figure 1 (success rate). If `random` or `bruteforce` reward
scalars are filled in `BASELINES`, they appear as horizontal bounds here too.

In [ ]:
"""Figure 3 — Shaped reward (secondary diagnostic; NOT primary evaluation)."""

def _resolve_baseline_reward(bval: dict, window: int = 10):
    if bval.get("reward") is not None:
        return float(bval["reward"])
    if bval.get("csv_path") is not None:
        p = Path(bval["csv_path"])
        if p.exists():
            df = pd.read_csv(p)
            if len(df) > 0 and "reward" in df.columns:
                return df["reward"].tail(window).mean()
    return None


fig, ax = plt.subplots(figsize=(11, 5.5))
any_plotted        = False
single_seed_warning = False

# llm_full lines (one per run)
for run in RUNS:
    sd = all_data[run["tag"]].get("llm_full", {})
    if not sd:
        continue
    eps, mu, lo, hi = mean_and_band(sd, "reward")
    lbl = run["label"]
    if len(sd) == 1:
        lbl += " (1 seed, rolling mean)"
        single_seed_warning = True
    ax.plot(eps, mu, color=run["color"], linestyle=run["ls"],
            linewidth=2.2, label=lbl, zorder=3)
    if lo is not None:
        ax.fill_between(eps, lo, hi, alpha=0.18, color=run["color"], zorder=2)
    any_plotted = True

# ppo_options (first run that has it)
for run in RUNS:
    sd = all_data[run["tag"]].get("ppo_options", {})
    if not sd:
        continue
    eps, mu, lo, hi = mean_and_band(sd, "reward")
    lbl = PPO_STYLE["label"]
    if len(sd) == 1:
        lbl += " (1 seed, rolling mean)"
        single_seed_warning = True
    ax.plot(eps, mu, color=PPO_STYLE["color"], linestyle=PPO_STYLE["ls"],
            linewidth=2.2, label=lbl, zorder=3)
    if lo is not None:
        ax.fill_between(eps, lo, hi, alpha=0.18, color=PPO_STYLE["color"], zorder=2)
    any_plotted = True
    break

# Baseline reward scalars
for bval in BASELINES.values():
    val = _resolve_baseline_reward(bval)
    if val is not None:
        ax.axhline(val, color=bval["color"], linestyle=bval["ls"],
                   linewidth=1.8, label=bval["label"], zorder=1)

ax.set_xlabel("Training Episode", fontsize=13, fontweight="bold")
ax.set_ylabel("Shaped Episode Reward", fontsize=13, fontweight="bold")
ax.set_title(
    f"RQ1 — Shaped Reward Over Training  [{SCENARIO} scenario]\n"
    "(diagnostic only — not the primary evaluation metric)",
    fontsize=14, fontweight="bold", pad=12)
ax.legend(loc="lower right", framealpha=0.9)
ax.axhline(0, color="black", linewidth=0.7, alpha=0.4, linestyle="--")
ax.text(0.01, 0.97,
        "⚠ Shaped reward — includes KL & step penalties that differ across conditions.\n"
        "Use Figure 1 (success rate) for cross-condition conclusions.",
        transform=ax.transAxes, fontsize=9, color="#8B0000", va="top", ha="left",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#FFF3F3",
                  edgecolor="#D55E00", alpha=0.9))
if single_seed_warning:
    ax.text(0.01, 0.01,
            "Note: only 1 seed — line is a 10-ep rolling mean, no CI.",
            transform=ax.transAxes, fontsize=9, color="#555555", va="bottom", ha="left")

plt.tight_layout()
save_figure(fig, "rq1_shaped_reward")
plt.show()

## 6. Export Verification

In [ ]:
"""Confirm all expected output files exist."""
expected = [
    "rq1_success_rate.png",       # Figure 1 — primary evaluation
    "rq1_success_rate.pdf",
    "rq1_sample_efficiency.png",  # Figure 2 — sample efficiency (success-rate threshold)
    "rq1_sample_efficiency.pdf",
    "rq1_shaped_reward.png",      # Figure 3 — diagnostic only
    "rq1_shaped_reward.pdf",
]
print("Export verification:")
all_ok = True
for fname in expected:
    p = FIGURES_DIR / fname
    status = "OK" if p.exists() else "MISSING"
    if status == "MISSING":
        all_ok = False
    print(f"  [{status}] {p}")

print()
if all_ok:
    print("All RQ1 figures exported successfully.")
else:
    print("Some figures are missing — re-run the plotting cells above.")

# Remind about unpopulated baselines
any_baseline_missing = any(
    b["success_rate"] is None and b["csv_path"] is None
    for b in BASELINES.values()
)
if any_baseline_missing:
    print()
    print("ACTION REQUIRED: Fill BASELINES dict in the configuration cell")
    print("with random-agent and bruteforce success_rate/reward values,")
    print("or csv_path pointers to their train.csv files.")